In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
df=pd.read_csv("housing.csv")
df.head()
df.info()
df.isnull().sum()
df.describe()
df['median_income_cat']=pd.cut(df["median_income"],bins=[0,1.5,3,4.5,6,np.inf],labels=[1,2,3,4,5]).copy()
split=StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index,test_index in split.split(df,df["median_income_cat"]):
    train_set=df.loc[train_index]
    test_set=df.loc[test_index]
test_set.info()
train_set.info()
housing=train_set.copy()
housing=housing.drop("median_income_cat",axis=1)
print(housing)
housing_labels=housing["median_house_value"].copy()
housing=housing.drop("median_house_value",axis=1)
housing=housing.reset_index(drop=True)
housing.isnull().sum()
housing_num=list(housing.select_dtypes(include=np.number).columns)
housing_cat=["ocean_proximity"]
num_pipeline=Pipeline([
    ("impute",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
cat_pipeline=Pipeline([
    ("OneHot",OneHotEncoder(handle_unknown="ignore"))
])
full_pipeline=ColumnTransformer([
    ("num",num_pipeline,housing_num),
    ("cat",cat_pipeline,housing_cat)
])
housing_prepared = full_pipeline.fit_transform(housing)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 4128 entries, 5241 to 3965
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           4128 non-null   float

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import  root_mean_squared_error
lin_reg=LinearRegression()
lin_reg.fit(housing_prepared,housing_labels)

LinearRegression()